# NyaayKhel — 00: Kabaddi Clip Downloader

**Purpose:** Download 150–300 short (2–10 sec) kabaddi clips from YouTube for training data.

**What this notebook produces:**
- `data/raw/<angle_bucket>/<clip_id>.mp4` — trimmed short clips, ready for labeling
- `data/raw/clip_index.csv` — metadata: clip ID, source URL, angle bucket, duration

**Ground rules (read before running):**
- All footage is from publicly available YouTube videos. Document source URLs in `clip_index.csv`.
- Target angle distribution: ~50% side-view (~90°), ~30% quarter-view (~70°), ~20% angled (~60°)
- Keep clips short (2–10 sec). Labeling is faster with short clips.
- Quality > quantity. 150 well-labeled clips > 300 noisy ones.

**Exit gate:** 150+ clips downloaded and organised into angle buckets.

## Cell 1: Install Dependencies

In [ ]:
# FIX (bot-detection): Use the latest yt-dlp nightly build.
# Stable releases often lag YouTube's anti-bot changes by weeks.
# --pre picks up nightly wheels that include the freshest cookie/signature patches.
!pip install -q -U --pre yt-dlp ffmpeg-python

# FIX (JS runtime): Install Deno so yt-dlp can execute YouTube's JS challenges.
# Without a JS runtime yt-dlp prints 'No supported JavaScript runtime could be found'
# and falls back to incomplete extraction, which triggers the bot-detection gate.
import subprocess, os
deno_result = subprocess.run(
    'curl -fsSL https://deno.land/install.sh | sh',
    shell=True, capture_output=True, text=True
)
if deno_result.returncode == 0:
    deno_bin = os.path.expanduser('~/.deno/bin')
    os.environ['PATH'] = deno_bin + ':' + os.environ.get('PATH', '')
    deno_ver = subprocess.run(['deno', '--version'], capture_output=True, text=True)
    print('deno:', deno_ver.stdout.split('\n')[0] if deno_ver.returncode == 0 else 'installed (restart kernel to pick up PATH)')
else:
    print('WARNING: deno install failed — yt-dlp may still work without it.')
    print(deno_result.stderr[-400:])

# Verify ffmpeg is available (Colab usually has it pre-installed)
result = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
print('ffmpeg:', result.stdout.split('\n')[0] if result.returncode == 0 else 'NOT FOUND — run: !apt-get install -y ffmpeg')

import yt_dlp
print(f'yt-dlp version: {yt_dlp.version.__version__}')

## Cell 1b: (Optional) Cookie Setup — Bypass Bot-Detection

If downloads still fail with *"Sign in to confirm you're not a bot"*:

1. Export your browser cookies for `youtube.com` using the [cookies.txt extension](https://addons.mozilla.org/en-US/firefox/addon/cookies-txt/) (Firefox) or [Get cookies.txt LOCALLY](https://chrome.google.com/webstore/detail/get-cookiestxt-locally/cclelndahbckbenkjhflpdbgdldlbecc) (Chrome).
2. Upload the file to `/content/cookies.txt` in Colab (Files panel on the left), **or** save it to your Drive as `NyaayKhel/cookies.txt` and adjust `COOKIES_PATH` below.
3. Run this cell — it sets `COOKIES_PATH` which Cell 4 picks up automatically.

In [ ]:
import os

# Paths to check for a cookies file, in order of preference.
# Adjust or add paths if you stored your cookies elsewhere.
_COOKIE_CANDIDATES = [
    '/content/cookies.txt',                          # uploaded directly to Colab
    '/content/drive/MyDrive/NyaayKhel/cookies.txt', # stored in Drive
]

COOKIES_PATH = None
for _p in _COOKIE_CANDIDATES:
    if os.path.exists(_p):
        COOKIES_PATH = _p
        print(f'✓ cookies.txt found: {COOKIES_PATH}')
        print('  Bot-detection bypass is ENABLED for Cell 4.')
        break

if COOKIES_PATH is None:
    print('ℹ  No cookies.txt found — downloads will proceed without cookie auth.')
    print('   If you hit bot-detection errors in Cell 4, upload cookies.txt and re-run this cell.')

## Cell 2: Mount Google Drive (optional — for persistent storage across sessions)

In [ ]:
# Mount Google Drive so clips persist across Colab sessions.
# Skip this cell if you want to store clips locally in Colab's /content/ (lost on disconnect).

USE_DRIVE = True  # Set False to store in /content/ only

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/NyaayKhel'
else:
    BASE_DIR = '/content/NyaayKhel'

import os
RAW_DIR = os.path.join(BASE_DIR, 'data', 'raw')
for bucket in ['side_90', 'quarter_70', 'angled_60', 'misc']:
    os.makedirs(os.path.join(RAW_DIR, bucket), exist_ok=True)

print(f'Storage base: {BASE_DIR}')
print(f'Raw clip dirs created under: {RAW_DIR}')

## Cell 3: Define Search Terms & Download Config

In [ ]:
import csv, os, time, hashlib
from pathlib import Path

# ---------------------------------------------------------------------------
# TARGET: 150–300 clips across 4 classes (raid_start, touch, escape_return, neutral)
# Each "video" below will be downloaded, then sliced into short clips.
# You'll label individual clips in CVAT (Phase B).
# ---------------------------------------------------------------------------

# YouTube search queries — yt-dlp supports 'ytsearch<N>:query' syntax
# Each query downloads N videos matching the search.
# We then slice each video into short clips.
SEARCH_QUERIES = [
    # Side-view (~90°) — most important for training
    ('ytsearch5:kabaddi match full tournament side view', 'side_90'),
    ('ytsearch5:kabaddi raid touch district tournament', 'side_90'),
    ('ytsearch5:kabaddi village match full video India', 'side_90'),
    ('ytsearch3:kabaddi grassroots match recording', 'side_90'),
    # Quarter-view (~60-70°)
    ('ytsearch4:kabaddi tournament state level match', 'quarter_70'),
    ('ytsearch3:pro kabaddi training footage', 'quarter_70'),
    # Misc angles — for robustness
    ('ytsearch3:kabaddi national championship full match', 'misc'),
    ('ytsearch3:kho kho kabaddi wrestling grassroots sport India', 'misc'),
]

# FIX (duration filter): The original filter kept videos UNDER 600 s (10 min),
# which rejected every full kabaddi match (20–90 min) — exactly the footage we
# need for slicing into 5 s clips.
#
# Corrected logic:
#   MIN_VIDEO_DURATION_SEC  — reject anything too short to contain a real raid
#                             sequence (< 3 min).  Clips and trailers slip through
#                             search but contain no sustained gameplay.
#   MAX_VIDEO_DURATION_SEC  — soft upper cap.  90 min covers broadcast matches;
#                             raise or set to None to accept longer archive footage.
MIN_VIDEO_DURATION_SEC = 180    # 3 min  — skip clips / trailers
MAX_VIDEO_DURATION_SEC = 5400   # 90 min — skip extremely long unrelated footage

# Clip slicing config (applied AFTER download — do not change)
CLIP_DURATION_SEC = 5    # Each output clip length
CLIP_STRIDE_SEC = 3      # Stride between clip starts (overlap = 2 sec)

print(f'Configured {len(SEARCH_QUERIES)} search queries')
print(f'Video duration filter: {MIN_VIDEO_DURATION_SEC}s – {MAX_VIDEO_DURATION_SEC}s')
print(f'Clip: {CLIP_DURATION_SEC}s | Stride: {CLIP_STRIDE_SEC}s')

## Cell 4: Download Videos

In [ ]:
import yt_dlp
import json

DOWNLOAD_DIR = os.path.join(BASE_DIR, 'data', 'raw', 'full_videos')
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

# Index file to track sources
INDEX_PATH = os.path.join(RAW_DIR, 'download_index.csv')
downloaded_videos = []  # will hold metadata dicts

# ── Skip-reason counters (for the summary at the end) ─────────────────────
skip_counts = {'too_short': 0, 'too_long': 0, 'bot_check': 0, 'other_error': 0}

# FIX (duration filter): match_filter now enforces a MINIMUM duration.
# Full kabaddi matches are 20–90 min; the old `duration < 600` filter
# was rejecting them all and only accepting short clips that then failed
# the bot check anyway.
def _duration_filter(info, *, incomplete):
    """Accept videos whose duration falls in [MIN, MAX]. Return None to accept,
    a rejection-reason string to skip."""
    dur = info.get('duration') or 0
    if dur and dur < MIN_VIDEO_DURATION_SEC:
        skip_counts['too_short'] += 1
        return f'Duration {dur}s < minimum {MIN_VIDEO_DURATION_SEC}s — skipping short clip/trailer'
    if dur and dur > MAX_VIDEO_DURATION_SEC:
        skip_counts['too_long'] += 1
        return f'Duration {dur}s > maximum {MAX_VIDEO_DURATION_SEC}s — skipping excessively long video'
    return None  # accept

# FIX (bot-detection): add cookiefile if one was found in Cell 1b.
# The cookiefile lets yt-dlp authenticate as a logged-in user, which
# bypasses the 'Sign in to confirm you\'re not a bot' gate.
ydl_opts = {
    'format': 'bestvideo[height<=720][ext=mp4]+bestaudio[ext=m4a]/best[height<=720][ext=mp4]/best[height<=720]',
    'outtmpl': os.path.join(DOWNLOAD_DIR, '%(id)s.%(ext)s'),
    'noplaylist': True,
    'quiet': False,
    'no_warnings': False,
    'match_filter': _duration_filter,
    'writeinfojson': True,  # saves <id>.info.json alongside each video
    'ignoreerrors': True,   # skip unavailable / bot-blocked videos
}

# Attach cookies only if the file actually exists (optional — won't crash if absent)
try:
    _cookies = COOKIES_PATH  # set in Cell 1b
except NameError:
    _cookies = None
if _cookies and os.path.exists(_cookies):
    ydl_opts['cookiefile'] = _cookies
    print(f'🍪 Using cookies from: {_cookies}')
else:
    print('ℹ  No cookies.txt in use — run Cell 1b to enable cookie auth if needed.')

print(f'Downloading to: {DOWNLOAD_DIR}')
print(f'Duration filter: {MIN_VIDEO_DURATION_SEC}s – {MAX_VIDEO_DURATION_SEC}s')
print('This will take several minutes depending on connection speed.\n')

for query, bucket in SEARCH_QUERIES:
    print(f'--- Query: "{query}" → bucket: {bucket} ---')
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(query, download=True)
            if info and 'entries' in info:
                for entry in info['entries']:
                    if entry is None:
                        continue
                    video_id = entry.get('id', 'unknown')
                    duration = entry.get('duration', 0)
                    title = entry.get('title', '')
                    webpage_url = entry.get('webpage_url', '')
                    downloaded_videos.append({
                        'video_id': video_id,
                        'title': title,
                        'url': webpage_url,
                        'duration_sec': duration,
                        'angle_bucket': bucket,
                        'file': os.path.join(DOWNLOAD_DIR, f'{video_id}.mp4'),
                    })
                    print(f'  ✓ {video_id} | {duration}s | {title[:60]}')
    except Exception as e:
        err_str = str(e)
        if 'Sign in' in err_str or 'bot' in err_str.lower():
            skip_counts['bot_check'] += 1
            print(f'  ✗ Bot-check blocked query "{query}" — upload cookies.txt to fix')
        else:
            skip_counts['other_error'] += 1
            print(f'  ✗ Failed query "{query}": {e}')
    time.sleep(1)  # polite delay between queries

# Write download index
if downloaded_videos:
    with open(INDEX_PATH, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=downloaded_videos[0].keys())
        writer.writeheader()
        writer.writerows(downloaded_videos)
    print(f'\nDownload index written to {INDEX_PATH}')

# ── Download summary ───────────────────────────────────────────────────────
print('\n' + '='*60)
print('DOWNLOAD SUMMARY')
print('='*60)
print(f'  Videos downloaded (accepted):  {len(downloaded_videos)}')
print(f'  Skipped — too short (<{MIN_VIDEO_DURATION_SEC}s):  {skip_counts["too_short"]}')
print(f'  Skipped — too long  (>{MAX_VIDEO_DURATION_SEC}s):  {skip_counts["too_long"]}')
print(f'  Skipped — bot-check blocked:   {skip_counts["bot_check"]}')
print(f'  Skipped — other error:         {skip_counts["other_error"]}')
print('='*60)
if skip_counts['bot_check'] > 0:
    print('⚠  Bot-check failures detected. Fix options (in order):')
    print('   1. Upload cookies.txt to /content/ and re-run Cell 1b + Cell 4')
    print('   2. Wait a few hours and retry (YouTube rate-limits by IP)')
    print('   3. Run the notebook from a different Colab runtime region')
if len(downloaded_videos) == 0:
    print('\n⛔ 0 videos downloaded — check skip reasons above before running Cell 5.')
else:
    print(f'\n✅ Ready for Cell 5 — {len(downloaded_videos)} video(s) to slice into clips.')

## Cell 5: Slice Videos into Short Clips

In [ ]:
import subprocess
import uuid

# ---------------------------------------------------------------------------
# Slice each downloaded video into CLIP_DURATION_SEC clips.
# Output naming: <video_id>_t<start_sec>.mp4
# Output goes into the angle-bucket folder for that video.
# ---------------------------------------------------------------------------

CLIP_INDEX_PATH = os.path.join(RAW_DIR, 'clip_index.csv')
clip_records = []
total_clips = 0
skipped = 0

for vid in downloaded_videos:
    src = vid['file']
    if not os.path.exists(src):
        print(f'Skip (file missing): {src}')
        skipped += 1
        continue

    bucket_dir = os.path.join(RAW_DIR, vid['angle_bucket'])
    duration = vid['duration_sec'] or 0
    if duration < 3:
        print(f'Skip (too short): {vid["video_id"]} ({duration}s)')
        skipped += 1
        continue

    t = 0
    while t + CLIP_DURATION_SEC <= duration:
        clip_id = f"{vid['video_id']}_t{int(t):05d}"
        out_path = os.path.join(bucket_dir, f'{clip_id}.mp4')

        if not os.path.exists(out_path):
            cmd = [
                'ffmpeg', '-y',
                '-ss', str(t),
                '-i', src,
                '-t', str(CLIP_DURATION_SEC),
                '-c:v', 'libx264',
                '-preset', 'fast',
                '-crf', '23',
                '-an',           # drop audio — not needed for pose extraction
                '-vf', 'scale=640:-2',  # downsample to 640px wide
                out_path
            ]
            result = subprocess.run(cmd, capture_output=True)
            if result.returncode != 0:
                print(f'  ffmpeg error on {clip_id}: {result.stderr[-200:]}')

        clip_records.append({
            'clip_id': clip_id,
            'source_video_id': vid['video_id'],
            'source_url': vid['url'],
            'start_sec': t,
            'duration_sec': CLIP_DURATION_SEC,
            'angle_bucket': vid['angle_bucket'],
            'file': out_path,
            'label': '',  # filled in during Phase B labeling
        })
        total_clips += 1
        t += CLIP_STRIDE_SEC

# Write clip index
with open(CLIP_INDEX_PATH, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=clip_records[0].keys() if clip_records else ['clip_id'])
    writer.writeheader()
    writer.writerows(clip_records)

print(f'Total clips generated: {total_clips}')
print(f'Skipped videos: {skipped}')
print(f'Clip index written to: {CLIP_INDEX_PATH}')

# Angle bucket summary
from collections import Counter
bucket_counts = Counter(r['angle_bucket'] for r in clip_records)
print('\nClips per angle bucket:')
for bucket, count in bucket_counts.items():
    print(f'  {bucket}: {count} clips')

## Cell 6: Quick Visual Review — Spot-Check Clips

In [ ]:
# Display a random sample of clips as thumbnails to sanity-check quality.
# Run this in Colab — it renders inline images.

import random
import subprocess
from IPython.display import Image, display

SAMPLE_N = min(12, len(clip_records))
sample = random.sample(clip_records, SAMPLE_N)

THUMB_DIR = '/tmp/thumbs'
os.makedirs(THUMB_DIR, exist_ok=True)

print(f'Showing {SAMPLE_N} random clip thumbnails (frame at 1 sec):\n')
for r in sample:
    if not os.path.exists(r['file']):
        continue
    thumb_path = os.path.join(THUMB_DIR, r['clip_id'] + '.jpg')
    subprocess.run([
        'ffmpeg', '-y', '-ss', '1', '-i', r['file'],
        '-frames:v', '1', '-q:v', '3', thumb_path
    ], capture_output=True)
    if os.path.exists(thumb_path):
        print(f"  {r['clip_id']} | bucket: {r['angle_bucket']}")
        display(Image(thumb_path, width=320))

print('\n--- Review complete ---')
print('MANUAL STEP: Delete clips that are blurry, not kabaddi, or wrong angle before labeling.')

## Cell 7: Prepare Clip List for CVAT / Label Studio

After reviewing clips above, use this cell to generate the import file for your labeling tool.

In [ ]:
# Reload clip index (in case you cleaned up bad clips manually)
import csv

with open(CLIP_INDEX_PATH, 'r', encoding='utf-8') as f:
    clips = list(csv.DictReader(f))


# Recover source URLs if this index was rebuilt without provenance.
# yt-dlp writes <video_id>.info.json sidecars in full_videos/ when writeinfojson=True.
def _is_missing_source_url(row):
    value = (row.get('source_url') or '').strip()
    return value == '' or value == 'https://youtube.com'

if any(_is_missing_source_url(c) for c in clips):
    info_dir = os.path.join(RAW_DIR, 'full_videos')
    url_by_video_id = {}
    if os.path.isdir(info_dir):
        for fname in os.listdir(info_dir):
            if not fname.endswith('.info.json'):
                continue
            try:
                with open(os.path.join(info_dir, fname), 'r', encoding='utf-8') as jf:
                    info = json.load(jf)
                video_id = info.get('id') or fname.replace('.info.json', '')
                source_url = info.get('webpage_url') or info.get('original_url') or info.get('url')
                if video_id and source_url:
                    url_by_video_id[video_id] = source_url
            except Exception as e:
                print(f'Could not read {fname}: {e}')

    recovered = 0
    for c in clips:
        if not _is_missing_source_url(c):
            continue
        video_id = c.get('source_video_id') or c.get('clip_id', '').split('_t')[0]
        if video_id in url_by_video_id:
            c['source_video_id'] = video_id
            c['source_url'] = url_by_video_id[video_id]
            recovered += 1

    if recovered:
        fieldnames = list(clips[0].keys())
        if 'source_url' not in fieldnames:
            fieldnames.append('source_url')
        if 'source_video_id' not in fieldnames:
            fieldnames.append('source_video_id')
        with open(CLIP_INDEX_PATH, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(clips)
        print(f'Recovered source URLs for {recovered} clip rows from .info.json sidecars.')
    else:
        print('WARNING: Some source URLs are missing and no .info.json sidecars recovered them.')
        print('Keep download_index.csv or full_videos/*.info.json so future clip_index.csv files retain provenance.')

# Filter to only clips that actually exist
valid_clips = [c for c in clips if os.path.exists(c['file'])]
print(f'Valid clips ready for labeling: {len(valid_clips)}')

# Export paths list for CVAT upload
paths_file = os.path.join(RAW_DIR, 'cvat_import_paths.txt')
with open(paths_file, 'w') as f:
    for c in valid_clips:
        f.write(c['file'] + '\n')

print(f'CVAT import paths written to: {paths_file}')
print()
print('LABEL CLASSES (use exactly these strings in CVAT/Label Studio):')
for cls in ['raid_start', 'touch', 'escape_return', 'neutral']:
    print(f'  • {cls}')
print()
print('Target distribution (rough guide):')
print('  raid_start:    ~30%')
print('  touch:         ~20%')
print('  escape_return: ~20%')
print('  neutral:       ~30%')
print()
print('→ Next step: Upload clips to CVAT/Label Studio and label each clip.')
print('  After labeling, export annotations as CSV or JSON and save to data/processed/')
print('  Then run notebook 02_dataset_builder.ipynb.')


## Exit Gate Checklist

Before moving to Phase B (labeling + model training):

- [ ] `data/raw/download_index.csv` exists with ≥ 15 source videos
- [ ] `data/raw/clip_index.csv` exists with ≥ 150 clip entries
- [ ] At least 100 clips actually exist on disk and pass visual review
- [ ] Clips are distributed across `side_90/`, `quarter_70/`, `angled_60/` buckets
- [ ] `data/raw/cvat_import_paths.txt` exported for labeling tool
- [ ] Notebook committed to GitHub with outputs cleared